In [ ]:
# --- vep path bootstrap (auto-added; safe to re-run) ---
import os, sys
from pathlib import Path
def _vep_root():
    cands = []
    _p = globals().get('__vsc_ipynb_file__')           # VSCode exposes the notebook path
    if _p: cands.append(Path(_p).resolve().parent)
    cands.append(Path.cwd().resolve())
    for s in cands:
        for c in [s, *s.parents]:                       # search upward for the repo root
            if (c / 'core').is_dir() and (c / 'classifiers').is_dir():
                return c
    return Path.cwd().resolve()
_root = _vep_root()
os.chdir(_root)                                          # data/ and outputs/ resolve from repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))                      # makes core/classifiers/app importable
# --- end vep path bootstrap ---

import os
from pathlib import Path

# Change working directory to parent directory
import sys
# Locate the repo root (folder containing the `core` package) and make it importable
_root = Path.cwd()
if not (_root / "core").is_dir():
    _root = _root.parent
os.chdir(_root)                       # data/ and outputs/ paths resolve from the repo root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))    # so `core`, `classifiers`, `app`, ... import in any kernel

# Optional: verify
print(Path.cwd())

import numpy as np 
import matplotlib.pyplot as plt
from core.preprocessing import extract_PulseWidth_SignalPower, preprocess_signal
import os
import re
import pandas as pd
from core import config
from scipy.signal import find_peaks
from core.helpers import load_dataset_paths, compute_average_signal, load_preprocessed_signal, get_data, load_blind_signal, load_signal
from core.preprocessing import preprocess_signal, preprocess_save_all, average_two_phases

In [ ]:
DEVICES = config.DEVICES
LABELS = ["BC_Only", "BC_and_RGC"]

data_dir = "data/Labelled_VEP_Data"
# print how many files we have for each device and label
for device in DEVICES:
    for label in LABELS:
        device_label_dir = os.path.join(data_dir, device, label)
        num_files = len(os.listdir(device_label_dir))
        print(f"{device} - {label}: {num_files} files")

In [ ]:
TMAX = config.TMAX
DELAY = 0

preprocess_save_all(
    normalize=True,
    tmin=-10,
    tmax=TMAX,
    SNR_threshold=1.0,
    include_blind=True,
    do_artifact_removal=False,
)

all_paths_preprocessed = load_dataset_paths()
blind_files_sorted = load_blind_signal()

for device in DEVICES:
    print(f"\n{device}:")
    for label in LABELS:
        print(f"{label}: {len(all_paths_preprocessed[device][label])}")

print(f"\nBLIND: {len(blind_files_sorted)}")

In [ ]:
# Example usage:
prima_files_BC_only = all_paths_preprocessed["PRIMA_LE_DA"]["BC_Only"]
prima_files_RGC_only = all_paths_preprocessed["PRIMA_LE_DA"]["RGC_Only"]
prima_files_BC_and_RGC = all_paths_preprocessed["PRIMA_LE_DA"]["BC_and_RGC"]

mp20_files_BC_only = all_paths_preprocessed["MP20_LE_DA"]["BC_Only"]
mp20_files_RGC_only = all_paths_preprocessed["MP20_LE_DA"]["RGC_Only"]
mp20_files_BC_and_RGC = all_paths_preprocessed["MP20_LE_DA"]["BC_and_RGC"]

# print how many files are in each category
print(f"PRIMA_LE_DA BC_Only files: {len(all_paths_preprocessed["PRIMA_LE_DA"]["BC_Only"])}")
print(f"PRIMA_LE_DA RGC_Only files: {len(all_paths_preprocessed["PRIMA_LE_DA"]["RGC_Only"])}")
print(f"PRIMA_LE_DA BC_and_RGC files: {len(all_paths_preprocessed["PRIMA_LE_DA"]["BC_and_RGC"])}")

print(f"MP20_LE_DA BC_Only files: {len(all_paths_preprocessed["MP20_LE_DA"]["BC_Only"])}")
print(f"MP20_LE_DA RGC_Only files: {len(all_paths_preprocessed["MP20_LE_DA"]["RGC_Only"])}")
print(f"MP20_LE_DA BC_and_RGC files: {len(all_paths_preprocessed["MP20_LE_DA"]["BC_and_RGC"])}")

## 1.1 Plotting Training Data

In [ ]:
# Colors for the two conditions
CONDITION_COLORS = {
    "BC_Only": "dodgerblue",
    "BC_and_RGC": "deeppink",
}


LABEL_FONTSIZE = 15

DEVICE_TITLES = {
    "PRIMA_LE_DA": "PRIMA LE DA",
    "PRIMA_RCS_DA": "PRIMA RCS DA",
    "MP20_LE_DA": "MP20 LE DA",
    "MP20_RCS_LA": "MP20 RCS LA",
    "RB20_RCS_LA": "RB20 RCS LA",
}

def parse_filename_info(filename):
    parts = os.path.splitext(os.path.basename(filename))[0].split("_")
    device = parts[0] if parts[0] not in ("LE", "ML") else (parts[0] if parts[1].isdigit() else parts[1])
    try:
        animal = int(parts[2] if parts[0] == "RCS" else (parts[1] if parts[1].isdigit() else parts[2]))
    except (ValueError, IndexError):
        animal = None
    return animal, device

def iter_csv_files(base_path, device_folder, conditions=("BC_Only", "BC_and_RGC")):
    for condition in conditions:
        condition_path = os.path.join(base_path, device_folder, condition)
        if not os.path.isdir(condition_path):
            continue
        for entry in os.scandir(condition_path):
            if entry.is_file() and entry.name.endswith(".csv"):
                animal, device = parse_filename_info(entry.name)
                if animal is not None:
                    pw, irr = extract_PulseWidth_SignalPower(entry.path)
                    yield {"filepath": entry.path, "animal_number": animal, "pulse_width": pw, 
                           "irradiance": irr, "condition": condition, "device": device}

def select_irradiance_samples(files, n):
    if not n or n <= 0 or n >= len(files):
        return files
    idxs = np.unique(np.round(np.linspace(0, len(files) - 1, n)).astype(int))
    if len(idxs) < n:
        candidates = [i for i in range(len(files)) if i not in set(idxs)]
        extra = np.round(np.linspace(0, len(candidates) - 1, n - len(idxs))).astype(int)
        idxs = np.sort(np.concatenate([idxs, [candidates[i] for i in extra]]))
    return [files[i] for i in idxs[:n]]

def plot_animal_traces(base_path, device_folder, target_pulse_width=10.0, tolerance=0.01, n_traces=None, vline_x=0.0):
    per_animal = {}
    for rec in iter_csv_files(base_path, device_folder):
        if abs(rec["pulse_width"] - target_pulse_width) < tolerance:
            per_animal.setdefault(rec["animal_number"], {}).setdefault(rec["irradiance"], rec)
    
    if not per_animal:
        return
    
    irr_map = max(per_animal.values(), key=len)
    files = select_irradiance_samples([irr_map[k] for k in sorted(irr_map)], n_traces)
    print(files)
    n = len(files)
    if not n:
        return

    title = DEVICE_TITLES.get(device_folder, device_folder)
    fig, axes = plt.subplots(n, 1, figsize=(7, n * 1.1), squeeze=False)
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)

    # Draw once, then after layout we'll position the "mW/mm²" header
    # centered over the y-labels. We use a placeholder and fix position after tight_layout.
    unit_label = fig.text(0, 0, "mW/mm²", fontsize=LABEL_FONTSIZE, ha="center", va="bottom")

    # First pass: preprocess and collect time limits
    preprocessed = []
    t_min, t_max = None, None
    for rec in files:
        t, s, _ = preprocess_signal(
            rec["filepath"],
            normalize=True,
            do_artifact_removal=True,
            tmax=400,
            do_dwt_downsampling=False,
        )
        preprocessed.append((t, s, rec))
        t_min = t[0] if t_min is None else min(t_min, t[0])
        t_max = t[-1] if t_max is None else max(t_max, t[-1])

    for i, (t, s, rec) in enumerate(preprocessed):
        ax = axes[i, 0]
        color = CONDITION_COLORS.get(rec["condition"], "black")
        ax.plot(t, s, color=color, lw=2.5)

        # Y-label: only the irradiance number (unit is in header)
        ax.set_ylabel(f"{rec['irradiance']:.2f}", rotation=0, labelpad=35,
                      va="center", fontsize=LABEL_FONTSIZE)

        yr = max(s.max() - s.min(), 1e-6)
        ax.set_ylim(s.min() - 0.15 * yr, s.max() + 0.15 * yr)
        ax.set_xlim(t_min, t_max)

        # Hide all spines except bottom for last plot
        for sp in ['top', 'right', 'left']:
            ax.spines[sp].set_visible(False)
        
        # Only show x-axis on the bottom plot
        if i == n - 1:
            # Calculate tick positions (0, 100, 200, ...)
            tick_start = 0
            tick_end = int(np.ceil(t_max / 100) * 100)
            tick_positions = np.arange(tick_start, tick_end + 1, 100)
            
            # Show bottom spine (axis line)
            ax.spines['bottom'].set_position(('outward', 10))
            ax.spines['bottom'].set_visible(True)
            
            ax.set_xticks(tick_positions)
            ax.set_xticklabels([str(int(x)) for x in tick_positions], fontsize=LABEL_FONTSIZE)
            ax.set_xlabel("Time (ms)", fontsize=LABEL_FONTSIZE)
            ax.tick_params(axis='x', length=5, direction='out')
        else:
            ax.spines['bottom'].set_visible(False)
            ax.set_xticks([])
        
        ax.set_yticks([])

    # Legend for conditions
    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], color=CONDITION_COLORS["BC_Only"], lw=2.5, label="BC"),
        Line2D([0], [0], color=CONDITION_COLORS["BC_and_RGC"], lw=2.5, label="BC + RGC"),
    ]
    fig.legend(handles=legend_handles, loc="upper right", fontsize=LABEL_FONTSIZE,
               frameon=False, bbox_to_anchor=(1.0, 1.0))

    plt.tight_layout(rect=[0.05, 0, 1, 0.99])
    plt.subplots_adjust(hspace=0.15)

    # Center "mW/mm²" header over the y-axis labels
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    fig_width = fig.get_window_extent(renderer).width

    ax0 = axes[0, 0]
    bbox = ax0.yaxis.label.get_window_extent(renderer)
    label_x_norm = (bbox.x0 + bbox.width / 2) / fig_width

    top_y = axes[0, 0].get_position().y1
    unit_label.set_position((label_x_norm, top_y + 0.01))
    unit_label.set_transform(fig.transFigure)

    plt.show()

base_path = "data/Labelled_VEP_Data"
plot_animal_traces(base_path, "PRIMA_LE_DA", target_pulse_width=10.0, n_traces=5, vline_x=15.0)
plot_animal_traces(base_path, "PRIMA_RCS_DA", target_pulse_width=10.0, n_traces=5, vline_x=15.0)
plot_animal_traces(base_path, "MP20_LE_DA", target_pulse_width=10.0, n_traces=5, vline_x=15.0)
plot_animal_traces(base_path, "MP20_RCS_LA", target_pulse_width=10.0, n_traces=5, vline_x=15.0)
plot_animal_traces(base_path, "RB20_RCS_LA", target_pulse_width=10.0, n_traces=5, vline_x=15.0)

In [ ]:
def plot_2x2_grid(base_path, device_folders, target_pulse_width=10.0, tolerance=0.01, n_traces=5):
    """
    Tightly packed 2x2 grid.
    - Units (mW/mm²) ONLY on top row.
    - Time labels/axis ONLY on bottom row.
    - Legend ONLY on top-right panel.
    - Stretched vertically with zero internal trace padding.
    """
    grid_positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

    # [Data collection logic - Unchanged]
    all_data = []
    for device_folder in device_folders:
        per_animal = {}
        for rec in iter_csv_files(base_path, device_folder):
            if abs(rec["pulse_width"] - target_pulse_width) < tolerance:
                per_animal.setdefault(rec["animal_number"], {}).setdefault(rec["irradiance"], rec)
        if not per_animal:
            all_data.append([])
            continue
        irr_map = max(per_animal.values(), key=len)
        files = select_irradiance_samples([irr_map[k] for k in sorted(irr_map)], n_traces)
        preprocessed = []
        for rec in files:
            t, s, _ = preprocess_signal(rec["filepath"], normalize=True, do_artifact_removal=False, tmin=-20,
                                        tmax=400, do_dwt_downsampling=False)
            preprocessed.append((t, s, rec))
        all_data.append(preprocessed)

    # --- ENHANCED STRETCH & TIGHTNESS ---
    n_rows_per_panel = n_traces
    fig = plt.figure(figsize=(14, n_rows_per_panel * 2.5)) # Vertical stretch

    # Reduced gaps between the 4 main panels
    outer_grid = fig.add_gridspec(2, 2, hspace=0.18, wspace=0.25)

    for panel_idx, (device_folder, preprocessed) in enumerate(zip(device_folders, all_data)):
        row, col = grid_positions[panel_idx]
        title = DEVICE_TITLES.get(device_folder, device_folder)

        if not preprocessed:
            ax = fig.add_subplot(outer_grid[row, col]); ax.axis("off")
            continue

        n = len(preprocessed)
        # hspace=0.0 removes white space between traces within a panel
        inner_grid = outer_grid[row, col].subgridspec(n, 1, hspace=0.0)

        t_min = min(t[0] for t, s, rec in preprocessed)
        t_max = max(t[-1] for t, s, rec in preprocessed)

        for i, (t, s, rec) in enumerate(preprocessed):
            ax = fig.add_subplot(inner_grid[i])
            color = CONDITION_COLORS.get(rec["condition"], "black")
            ax.plot(t, s, color=color, lw=2.5) # Thicker line for visibility

            ax.set_ylabel(f"{rec['irradiance']:.2f}", rotation=0, labelpad=30,
                          va="center", fontsize=LABEL_FONTSIZE - 1)

            yr = max(s.max() - s.min(), 1e-6)
            ax.set_ylim(s.min() - 0.05 * yr, s.max() + 0.05 * yr)
            ax.set_xlim(t_min, t_max)

            # Clean "no-box" look
            for sp in ['top', 'right', 'left', 'bottom']:
                ax.spines[sp].set_visible(False)
            ax.set_yticks([])
            ax.set_xticks([])

            # --- TOP TRACE OF PANEL: Title, Units, and Legend ---
            if i == 0:
                ax.set_title(title, fontsize=15, fontweight="bold", pad=12)
                
                # Unit label ONLY for the top row panels
                if row == 0:
                    ax.text(-0.12, 1.25, "mW/mm²", transform=ax.transAxes,
                            fontsize=LABEL_FONTSIZE, ha="center", va="bottom")

                # Legend ONLY for the top-right panel (panel_idx 1)
                if panel_idx == 1:
                    from matplotlib.lines import Line2D
                    legend_handles = [
                        Line2D([0], [0], color=CONDITION_COLORS["BC_Only"], lw=3, label="BC"),
                        Line2D([0], [0], color=CONDITION_COLORS["BC_and_RGC"], lw=3, label="BC + RGC"),
                    ]
                    ax.legend(handles=legend_handles, fontsize=LABEL_FONTSIZE - 1,
                              frameon=False, loc="lower left", bbox_to_anchor=(0.7, 1.05),
                              ncol=1, borderaxespad=0, handlelength=1.2)

            # --- BOTTOM TRACE OF PANEL: Time Axis ONLY on bottom row ---
            if i == n - 1 and row == 1:
                tick_positions = np.arange(0, int(np.ceil(t_max / 100) * 100) + 1, 100)
                ax.spines['bottom'].set_visible(True)
                ax.spines['bottom'].set_position(('outward', 4))
                ax.set_xticks(tick_positions)
                ax.set_xticklabels([str(int(x)) for x in tick_positions], fontsize=LABEL_FONTSIZE - 1)
                ax.set_xlabel("Time (ms)", fontsize=LABEL_FONTSIZE - 1)

    plt.savefig("outputs/figures/Data_example.png", dpi=300, bbox_inches="tight")
    plt.show()

plot_2x2_grid(
    base_path="data/Labelled_VEP_Data",
    device_folders=["PRIMA_LE_DA", "PRIMA_RCS_DA", "MP20_LE_DA", "MP20_RCS_LA"],
    target_pulse_width=10.0,
    n_traces=5,
)

In [ ]:
base_path = "data/Labelled_VEP_Data"
for device in DEVICES:
    print(f"\n{device}:")
    pulse_widths = []
    irradiances = []
    for label in LABELS:
        dir_path = os.path.join(base_path, device, label)
        # find the min and max pulse width and irradiance 
        for file in os.listdir(dir_path):
            if file.endswith(".csv"):
                filepath = os.path.join(dir_path, file)
                pw, irr = extract_PulseWidth_SignalPower(filepath)
                pulse_widths.append(pw)
                irradiances.append(irr)
    print(f"Pulse Widths: {min(pulse_widths):.2f} ms to {max(pulse_widths):.2f} ms")
    print(f"Irradiances: {min(irradiances):.2f} mW/mm² to {max(irradiances):.2f} mW/mm²")

## 1.3 Plotting example files

In [ ]:
# File paths
bc_example_file = "data/example_files/BC_only_1.09mWmm^2.csv"
rgc_example_file = "data/example_files/RGC_only_2.1mWmm^2.csv"
bc_rgc_example_file = "data/example_files/BC_&_RGC_1.39mWmm^2.csv"

# Load all three signals
time_bc_raw, signal_bc_raw = load_signal(bc_example_file)
time_bc, signal_bc = average_two_phases(time_bc_raw, signal_bc_raw)
time_bc = time_bc - time_bc[0]

time_rgc_raw, signal_rgc_raw = load_signal(rgc_example_file)
time_rgc, signal_rgc = average_two_phases(time_rgc_raw, signal_rgc_raw)
time_rgc = time_rgc - time_rgc[0]

time_bc_rgc_raw, signal_bc_rgc_raw = load_signal(bc_rgc_example_file)
time_bc_rgc, signal_bc_rgc = average_two_phases(time_bc_rgc_raw, signal_bc_rgc_raw)
time_bc_rgc = time_bc_rgc - time_bc_rgc[0]

In [ ]:
def add_time_scale_bar(ax, time_array, bar_length=100):
    """Add a clean scale bar for time reference"""
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    # Position at bottom right
    x_end = xlim[1] - (xlim[1] - xlim[0]) * 0.02
    x_start = x_end - bar_length
    y_bar = ylim[0] + (ylim[1] - ylim[0]) * 0.08
    
    # Draw scale bar
    ax.plot([x_start, x_end], [y_bar, y_bar], 'k-', lw=3)
    ax.plot([x_start, x_start], [y_bar - (ylim[1]-ylim[0])*0.02, 
            y_bar + (ylim[1]-ylim[0])*0.02], 'k-', lw=3)
    ax.plot([x_end, x_end], [y_bar - (ylim[1]-ylim[0])*0.02, 
            y_bar + (ylim[1]-ylim[0])*0.02], 'k-', lw=3)
    
    # Label
    ax.text((x_start + x_end) / 2, y_bar - (ylim[1]-ylim[0])*0.05, 
            f'{bar_length} ms', ha='center', va='top', 
            fontsize=20, fontweight='bold')

def find_peaks_and_valleys(time, signal, time_window=(0, 150), 
                           pos_prominence_pct=30, neg_prominence_pct=30,
                           distance_ms=5, pos_after_first_neg=False):
    """
    Automatically find peaks (positive) and valleys (negative) in signal.
    
    Parameters:
    -----------
    time : array
        Time array in ms
    signal : array
        Signal values
    time_window : tuple
        (start_time, end_time) in ms to search for peaks
    pos_prominence_pct : float
        Percentage of signal range for positive peak prominence threshold
    neg_prominence_pct : float
        Percentage of signal range for negative peak prominence threshold
    distance_ms : float
        Minimum distance between peaks in ms
    pos_after_first_neg : bool
        If True, only keep positive peaks that come after the first negative peak
    
    Returns:
    --------
    positive_peaks : array of indices
    negative_peaks : array of indices
    """
    # Find indices within time window
    mask = (time >= time_window[0]) & (time <= time_window[1])
    search_indices = np.where(mask)[0]
    
    if len(search_indices) == 0:
        return np.array([]), np.array([])
    
    # Calculate distance in samples
    avg_sample_rate = len(time) / (time[-1] - time[0])  # samples per ms
    distance_samples = int(distance_ms * avg_sample_rate)
    
    # Calculate prominence thresholds
    signal_range = np.max(signal[mask]) - np.min(signal[mask])
    pos_prominence = signal_range * (pos_prominence_pct / 100.0)
    neg_prominence = signal_range * (neg_prominence_pct / 100.0)
    
    # Find negative peaks first (minima) by inverting signal
    negative_peaks, _ = find_peaks(-signal[mask], 
                                    prominence=neg_prominence,
                                    distance=distance_samples)
    negative_peaks = search_indices[negative_peaks]
    
    # Find positive peaks (maxima)
    positive_peaks, _ = find_peaks(signal[mask], 
                                    prominence=pos_prominence,
                                    distance=distance_samples)
    positive_peaks = search_indices[positive_peaks]
    
    # Filter positive peaks if requested
    if pos_after_first_neg and len(negative_peaks) > 0:
        first_neg_time = time[negative_peaks[0]]
        positive_peaks = positive_peaks[time[positive_peaks] > first_neg_time]
    
    return positive_peaks, negative_peaks



# --- Setup Figure ---
fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=False)
plt.subplots_adjust(hspace=0.4)

# Data mapping for iteration
datasets = [
    {
        "time": time_bc, "signal": signal_bc, "title": "BC stimulation", 
        "color": "dodgerblue", "window": (0, 120), "pos_p": 15, "neg_p": 15, "dist": 8
    },
    {
        "time": time_rgc, "signal": signal_rgc, "title": "RGC stimulation", 
        "color": "orangered", "window": (0, 80), "pos_p": 40, "neg_p": 10, "dist": 5
    },
    {
        "time": time_bc_rgc, "signal": signal_bc_rgc, "title": "BC & RGC stimulation", 
        "color": "hotpink", "window": (0, 150), "pos_p": 10, "neg_p": 15, "dist": 5
    }
]

for i, data in enumerate(datasets):
    ax = axes[i]
    t, s = data["time"], data["signal"]
    
    # Plot raw signal
    ax.plot(t, s, color=data["color"], lw=4)
    
    # Find peaks and valleys using your existing function logic
    pos_peaks, neg_peaks = find_peaks_and_valleys(
        t, s, time_window=data["window"],
        pos_prominence_pct=data["pos_p"],
        neg_prominence_pct=data["neg_p"],
        distance_ms=data["dist"],
        pos_after_first_neg=True
    )
    
    # Sort peaks
    pos_peaks = pos_peaks[np.argsort(t[pos_peaks])]
    neg_peaks = neg_peaks[np.argsort(t[neg_peaks])]
    
    # Get y-limits for text offsetting
    ylim = (np.min(s), np.max(s))
    y_range = ylim[1] - ylim[0]
    ax.set_ylim(ylim[0] - 0.2*y_range, ylim[1] + 0.2*y_range) # Add padding for labels

    # Plot Positive Peaks (P1, P2...)
    for idx_p, p_idx in enumerate(pos_peaks, 1):
        ax.plot(t[p_idx], s[p_idx], 'o', color='black', markersize=6)
        ax.text(t[p_idx], s[p_idx] + (y_range * 0.08), f'P{idx_p}', 
                fontsize=18, fontweight='bold', va='bottom', ha='center')

    # Plot Negative Peaks (N1, N2...)
    for idx_n, n_idx in enumerate(neg_peaks, 1):
        ax.plot(t[n_idx], s[n_idx], 'o', color='black', markersize=6)
        ax.text(t[n_idx], s[n_idx] - (y_range * 0.12), f'N{idx_n}', 
                fontsize=18, fontweight='bold', va='top', ha='center')

    # Styling
    ax.set_title(data["title"], fontsize=24, fontweight="bold", loc='center')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    
    # Add scale bar ONLY to the bottom subplot
    if i == 2:
        add_time_scale_bar(ax, t, bar_length=100)

plt.tight_layout()
plt.savefig('outputs/figures/Combined_VEP_Stimulation.png', dpi=300, bbox_inches='tight')
plt.show()